## **Fault Tolerance and Execution Using Persistance**

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from typing import TypedDict
import time

In [2]:
## Define the state
class CrashState(TypedDict):
    ## Initialise the variables which will be used
    input: str
    step1: str
    step2: str
    step3: str

In [3]:
## Defining step1
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

In [4]:
## Defining step 2
def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

In [5]:
## Defining step3
def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [6]:
## Building graph
graph = StateGraph(CrashState)


## Add Nodes
graph.add_node("step_1", step_1)
graph.add_node("step_2", step_2)
graph.add_node("step_3", step_3)


## Add edges
graph.add_edge(START, "step_1")
graph.add_edge("step_1", "step_2")
graph.add_edge("step_2", "step_3")
graph.add_edge("step_3", END)


## Creating the checkpoint
checkpoint = InMemorySaver()


## Compiling the graph
builder = graph.compile(checkpointer=checkpoint)

In [7]:
from IPython.display import display, Markdown

mermaid_code = builder.get_graph().draw_mermaid()

display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	step_1(step_1)
	step_2(step_2)
	step_3(step_3)
	__end__([<p>__end__</p>]):::last
	__start__ --> step_1;
	step_1 --> step_2;
	step_2 --> step_3;
	step_3 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
## Executing the graph
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    builder.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [9]:
builder.get_state({"configurable": {"thread_id": "thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2b2-aa8d-621b-8001-9ecae4cc47be'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-11-27T06:24:14.539625+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2b2-aa85-6cd6-8000-1145fd0011d7'}}, tasks=(PregelTask(id='129a65ba-b530-4fe5-467e-d83632ddcf0d', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [10]:
list(builder.get_state_history({"configurable": {"thread_id": "thread-1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2b2-aa8d-621b-8001-9ecae4cc47be'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-11-27T06:24:14.539625+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2b2-aa85-6cd6-8000-1145fd0011d7'}}, tasks=(PregelTask(id='129a65ba-b530-4fe5-467e-d83632ddcf0d', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2b2-aa85-6cd6-8000-1145fd0011d7'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-11-27T06:24:14.536623+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'ch

In [12]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = builder.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [13]:
builder.get_state({"configurable": {"thread_id": "thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2bc-163f-6d5d-8003-fedf09305f14'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-11-27T06:28:27.424495+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2bc-163f-6d5c-8002-236117482548'}}, tasks=(), interrupts=())

In [15]:
list(builder.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2bc-163f-6d5d-8003-fedf09305f14'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-11-27T06:28:27.424495+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2bc-163f-6d5c-8002-236117482548'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2bc-163f-6d5c-8002-236117482548'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-27T06:28:27.424495+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ea2b2-aa8d-621b-8001-9ecae4cc47be'}}, tasks=(PregelTask(id='4b9e3a3f-08f7-30ae-2623-45c21b07190